# Data preparation

We build the three tables that every later notebook reads:

1. `data/processed/volcano_daily.csv`: one row per date, 2024-01-01 to 2026-10-21 (1,025 rows)
2. `data/processed/barangays_study_area.csv`: one row per barangay with centroid within 30 km of the crater (216 rows)
3. `data/processed/barangay_daily_panel.csv`: barangay x date panel (216 x 1,025 = 221,400 rows)

Joins use PSGC codes, lag features look backward only, and nothing from 2026 is used to fill or scale values.

## Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "requirements.txt").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src import geo

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
REPORT_LOG = ROOT / "data" / "report_log" / "report_log.csv"
PROCESSED.mkdir(parents=True, exist_ok=True)

START = pd.Timestamp("2024-01-01")
END = pd.Timestamp("2026-10-21")
TRAIN_END = pd.Timestamp("2025-12-31")
DATES = pd.date_range(START, END, freq="D", name="date")
STUDY_RADIUS_KM = 30

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)


def show(df, name):
    """Print row count and missing values per column."""
    print(f"{name}: {len(df):,} rows x {df.shape[1]} columns")
    na = df.isna().sum()
    na = na[na > 0]
    print("  missing:", "none" if na.empty else na.to_dict())


print("dates:", len(DATES), DATES[0].date(), "to", DATES[-1].date())

dates: 1025 2024-01-01 to 2026-10-21


## SO2 emission (t/d)

We combine the three SO2 files. Timestamps that appear in both the FlySpec and the mobile DOAS file with different
values stay separate readings, and days without a reading stay empty, not zero.

In [2]:
SO2_FILES = {
    "flyspec": "SO2 Kanlaon FlySpec Survey.csv",
    "mobile_doas": "SO2 Kanlaon Mobile DOAS Survey.csv",
    "codcod_scan_doas": "SO2 Kanlaon Codcod Scan DOAS.csv",
}
parts = []
for source, fname in SO2_FILES.items():
    df = pd.read_csv(RAW / fname)
    df["source"] = source
    show(df, fname)
    parts.append(df)

so2_raw = pd.concat(parts, ignore_index=True)
so2_raw["datetime"] = pd.to_datetime(so2_raw["Datetime"])
so2_raw["date"] = so2_raw["datetime"].dt.normalize()
print("species:", list(so2_raw["species"].unique()), "| units:", list(so2_raw["units"].unique()))
show(so2_raw, "so2_raw (combined)")

so2_raw = so2_raw[so2_raw["date"].between(START, END)]
show(so2_raw, "so2_raw (study period)")
print("rows sharing a timestamp with another file:", so2_raw.duplicated("datetime", keep=False).sum())

SO2 Kanlaon FlySpec Survey.csv: 335 rows x 7 columns
  missing: none
SO2 Kanlaon Mobile DOAS Survey.csv: 555 rows x 7 columns
  missing: none
SO2 Kanlaon Codcod Scan DOAS.csv: 71 rows x 7 columns
  missing: {'deviation': 70, 'scan': 71}
species: ['SO2'] | units: ['t/d']
so2_raw (combined): 961 rows x 9 columns
  missing: {'deviation': 70, 'scan': 71}
so2_raw (study period): 961 rows x 9 columns
  missing: {'deviation': 70, 'scan': 71}
rows sharing a timestamp with another file: 34


In [3]:
so2_daily = (so2_raw.groupby("date")["emission"]
             .agg(so2_mean="mean", so2_max="max", so2_n="count")
             .reindex(DATES))
so2_daily["so2_n"] = so2_daily["so2_n"].fillna(0).astype(int)
so2_daily["so2_observed"] = (so2_daily["so2_n"] > 0).astype(int)

# last observed daily mean, carried forward, and days since that reading (0 on observed days)
so2_daily["so2_last"] = so2_daily["so2_mean"].ffill()
last_obs_date = DATES.to_series().where(so2_daily["so2_observed"] == 1).ffill()
so2_daily["so2_days_since"] = (DATES.to_series() - last_obs_date).dt.days

show(so2_daily, "so2_daily")
print("days with SO2:", so2_daily["so2_observed"].sum())
so2_daily.describe().round(1)

so2_daily: 1,025 rows x 6 columns
  missing: {'so2_mean': 170, 'so2_max': 170}
days with SO2: 855


,so2_mean,so2_max,so2_n,so2_observed,so2_last,so2_days_since
count,855.0,855.0,1025.0,1025.0,1025.0,1025.0
mean,2410.5,2425.8,0.9,0.8,2352.8,0.3
std,1767.0,1763.1,1.0,0.4,1846.1,0.9
min,0.0,0.0,0.0,0.0,0.0,0.0
25%,1253.2,1271.5,1.0,1.0,1170.0,0.0
50%,1856.0,1878.0,1.0,1.0,1787.0,0.0
75%,2979.0,3000.0,1.0,1.0,2824.0,0.0
max,11556.0,11556.0,18.0,1.0,11556.0,6.0


## Volcanic earthquakes

On a date that is in the file, a classification with no row counts as 0. Dates with no rows at all stay empty and
get `eq_reported = 0`.

In [4]:
eq_raw = pd.read_csv(RAW / "Daily Count by Volcano.csv")
show(eq_raw, "eq_raw")
eq_raw["date"] = pd.to_datetime(eq_raw["DateTime"])
eq_raw = eq_raw[eq_raw["date"].between(START, END)]
show(eq_raw, "eq_raw (study period)")
print(eq_raw["Classification"].value_counts().to_string())

EQ_COLUMNS = {
    "VT": "eq_vt", "TQ": "eq_tq", "tele": "eq_tele", "LFVQ": "eq_lfvq",
    "Tremor": "eq_tremor", "E": "eq_e", "Lahar": "eq_lahar", "HFVQ(S)": "eq_hfvqs",
}
unknown = set(eq_raw["Classification"]) - set(EQ_COLUMNS)
assert not unknown, f"unmapped classifications: {unknown}"

eq_raw: 2,749 rows x 3 columns
  missing: none
eq_raw (study period): 2,747 rows x 4 columns
  missing: none
Classification
TQ         939
VT         933
tele       599
LFVQ       166
Tremor      98
E            8
Lahar        3
HFVQ(S)      1


In [5]:
eq_daily = (eq_raw.pivot_table(index="date", columns="Classification", values="Recorded", aggfunc="sum")
            .rename(columns=EQ_COLUMNS)
            .reindex(columns=list(EQ_COLUMNS.values()))
            .fillna(0))
eq_daily["eq_total"] = eq_daily.sum(axis=1)
eq_daily = eq_daily.reindex(DATES)
eq_daily["eq_reported"] = eq_daily["eq_total"].notna().astype(int)

show(eq_daily, "eq_daily")
print("dates with no earthquake record:", [d.date().isoformat() for d in eq_daily.index[eq_daily["eq_reported"] == 0]])

eq_daily: 1,025 rows x 10 columns
  missing: {'eq_vt': 8, 'eq_tq': 8, 'eq_tele': 8, 'eq_lfvq': 8, 'eq_tremor': 8, 'eq_e': 8, 'eq_lahar': 8, 'eq_hfvqs': 8, 'eq_total': 8}
dates with no earthquake record: ['2024-01-12', '2024-01-22', '2024-01-26', '2024-01-27', '2024-01-29', '2024-03-10', '2025-11-11', '2026-08-12']


## Plume (steaming activity)

- `Height_meter` is the plume height above the crater. Rows with intensity "obscured" record 0 m because the crater
  was not visible, so their height is missing, not zero.
- Intensity becomes an ordered score, with the midpoint for ranges such as "weak-moderate". "obscured" and
  "ash plume" get no score, and ash plumes get their own flag.
- Drift, the direction the plume moves toward, becomes degrees and then one of 8 sectors.

In [6]:
steam_raw = pd.read_csv(RAW / "Kanlaon Steaming Activity.csv")
show(steam_raw, "steam_raw")
steam_raw["datetime"] = pd.to_datetime(steam_raw["Date_Observation"])
steam_raw["date"] = steam_raw["datetime"].dt.normalize()
steam_raw = steam_raw[steam_raw["date"].between(START, END)].sort_values("datetime")
show(steam_raw, "steam_raw (study period)")

steam_raw: 3,221 rows x 6 columns
  missing: {'Point_of_Observation': 7, 'Source': 8, 'Intensity': 16, 'Height_meter': 4, 'Drift': 698}
steam_raw (study period): 3,221 rows x 8 columns
  missing: {'Point_of_Observation': 7, 'Source': 8, 'Intensity': 16, 'Height_meter': 4, 'Drift': 698}


In [7]:
INTENSITY_SCORE = {
    "no steaming": 0,
    "wispy-no steaming": 0.5,
    "very weak-no steaming": 0.5,
    "wispy": 1,
    "very weak": 1,
    "very weak-weak": 1.5,
    "weak": 2,
    "weak-moderate": 2.5,
    "moderate": 3,
    "moderate-voluminous": 3.5,
    "voluminous": 4,
    "obscured": np.nan,
    "ash plume": np.nan,
}
unknown = set(steam_raw["Intensity"].dropna()) - set(INTENSITY_SCORE)
assert not unknown, f"unmapped intensity labels: {unknown}"

steam_raw["intensity_score"] = steam_raw["Intensity"].map(INTENSITY_SCORE)
steam_raw["plume_height"] = steam_raw["Height_meter"].where(steam_raw["Intensity"] != "obscured")

print(steam_raw.groupby("Intensity", dropna=False)
      .agg(n=("datetime", "size"), score=("intensity_score", "first"),
           median_height=("plume_height", "median"))
      .sort_values("score").to_string())
show(steam_raw, "steam_raw (with intensity score and height)")

                          n  score  median_height
Intensity                                        
no steaming              10    0.0            0.0
very weak-no steaming    10    0.5            0.0
wispy-no steaming         2    0.5            0.0
very weak                68    1.0            0.0
wispy                    15    1.0           25.0
very weak-weak           55    1.5          100.0
weak                    448    2.0          100.0
weak-moderate           364    2.5          150.0
moderate               1140    3.0          200.0
moderate-voluminous     266    3.5          550.0
voluminous              205    4.0          700.0
ash plume                 3    NaN          300.0
obscured                619    NaN            NaN
NaN                      16    NaN           50.0
steam_raw (with intensity score and height): 3,221 rows x 10 columns
  missing: {'Point_of_Observation': 7, 'Source': 8, 'Intensity': 16, 'Height_meter': 4, 'Drift': 698, 'intensity_score': 638, 'plum

In [8]:
steam_raw["drift_deg_raw"] = steam_raw["Drift"].map(geo.drift_label_to_deg)
steam_raw["drift_sector"] = geo.deg_to_sector(steam_raw["drift_deg_raw"])

drift_map = (steam_raw.groupby("Drift", dropna=False)
             .agg(n=("datetime", "size"), degrees=("drift_deg_raw", "first"), sector=("drift_sector", "first"))
             .sort_values("n", ascending=False))
print("Raw drift label -> degrees -> 8-sector label")
print(drift_map.to_string())

unmapped = steam_raw["Drift"].notna() & steam_raw["drift_sector"].isna()
assert unmapped.sum() == 0, steam_raw.loc[unmapped, "Drift"].unique()

Raw drift label -> degrees -> 8-sector label
                     n  degrees sector
Drift                                 
SW                 870    225.0     SW
NaN                698      NaN    NaN
W                  365    270.0      W
NW                 234    315.0     NW
SE                 179    135.0     SE
NE                 162     45.0     NE
S                  141    180.0      S
WSW                118    247.5      W
SSW                 77    202.5     SW
general west        48    270.0      W
N                   47      0.0      N
E                   46     90.0      E
WNW                 45    292.5     NW
general east        39     90.0      E
NNE                 32     22.5     NE
general south       29    180.0      S
SSE                 19    157.5      S
ENE                 18     67.5      E
NNW                 17    337.5      N
ESE                 15    112.5     SE
general north       14      0.0      N
N-NE                 2     22.5     NE
NW/SW              

In [9]:
def mode_latest(s):
    """Most common value in the day; ties go to the one observed latest."""
    s = s.dropna()
    if s.empty:
        return None
    counts = s.value_counts()
    top = counts[counts == counts.max()].index
    return s[s.isin(top)].iloc[-1]


plume_daily = steam_raw.groupby("date").agg(
    plume_n_obs=("datetime", "size"),
    plume_height_max=("plume_height", "max"),
    plume_height_median=("plume_height", "median"),
    plume_intensity_max=("intensity_score", "max"),
    plume_intensity_mean=("intensity_score", "mean"),
    ash_plume_obs=("Intensity", lambda s: int((s == "ash plume").any())),
    drift_obs_n=("drift_sector", "count"),
    drift_obs_sector=("drift_sector", mode_latest),
).reindex(DATES)
plume_daily[["plume_n_obs", "ash_plume_obs", "drift_obs_n"]] = (
    plume_daily[["plume_n_obs", "ash_plume_obs", "drift_obs_n"]].fillna(0).astype(int))

drift_rad = np.radians(plume_daily["drift_obs_sector"].map(geo.SECTOR_DEG))
plume_daily["drift_obs_sin"] = np.sin(drift_rad)
plume_daily["drift_obs_cos"] = np.cos(drift_rad)

ties = steam_raw.dropna(subset=["drift_sector"]).groupby("date")["drift_sector"].agg(
    lambda s: (s.value_counts() == s.value_counts().max()).sum() > 1)
show(plume_daily, "plume_daily")
print("days with an observed drift:", plume_daily["drift_obs_sector"].notna().sum(),
      "| days where the daily mode was a tie (latest observation used):", int(ties.sum()))
print(plume_daily["drift_obs_sector"].value_counts().to_string())

plume_daily: 1,025 rows x 10 columns
  missing: {'plume_height_max': 142, 'plume_height_median': 142, 'plume_intensity_max': 144, 'plume_intensity_mean': 144, 'drift_obs_sector': 151, 'drift_obs_sin': 151, 'drift_obs_cos': 151}
days with an observed drift: 874 | days where the daily mode was a tie (latest observation used): 98
drift_obs_sector
SW    298
W     161
NW    102
NE     88
S      77
SE     76
E      51
N      21


## Wind (Open-Meteo, hourly at the crater)

- We use 06:00 to 18:00 Asia/Manila time, when plumes are observed and people are outdoors, and average directions
  as u/v components, never as raw degrees.
- Plume-level wind: each hour, u and v are interpolated linearly between the 800 and 700 hPa geopotential heights
  at 2,435 m (summit) plus that day's median plume height, clipped to that range (no extrapolation). Days without a
  plume height use the 2024-2025 median.
- `wind_drift_deg = (wind direction + 180) % 360` is the direction the plume is pushed toward.

In [10]:
wind_raw = pd.read_csv(RAW / "wind_kanlaon_hourly.csv")
show(wind_raw, "wind_raw")
wind_raw["time"] = pd.to_datetime(wind_raw["time"])
wind_raw["date"] = wind_raw["time"].dt.normalize()
print("wind time range:", wind_raw["time"].min(), "to", wind_raw["time"].max())

wind_day = wind_raw[wind_raw["time"].dt.hour.between(6, 18) & wind_raw["date"].between(START, END)].copy()
show(wind_day, "wind_day (06:00-18:00, study period)")
print("hours per day:", wind_day.groupby("date").size().value_counts().to_dict())

wind_raw: 24,576 rows x 9 columns
  missing: none
wind time range: 2024-01-01 00:00:00 to 2026-10-20 23:00:00
wind_day (06:00-18:00, study period): 13,312 rows x 10 columns
  missing: none
hours per day: {13: 1024}


In [11]:
train_plume_height = plume_daily.loc[:TRAIN_END, "plume_height_median"].median()
print(f"2024-2025 median plume height used to fill missing days: {train_plume_height:.0f} m")

plume_alt = (geo.CRATER_ELEV_M + plume_daily["plume_height_median"].fillna(train_plume_height)).rename("plume_alt_m")
wind_day = wind_day.merge(plume_alt, left_on="date", right_index=True, how="left")
show(wind_day, "wind_day (with plume altitude)")

for level in ["10m", "800hPa", "700hPa"]:
    wind_day[f"u_{level}"], wind_day[f"v_{level}"] = geo.wind_to_uv(
        wind_day[f"wind_speed_{level}"], wind_day[f"wind_direction_{level}"])

z800, z700 = wind_day["geopotential_height_800hPa"], wind_day["geopotential_height_700hPa"]
wind_day["u_plume"], wind_day["weight_700hPa"] = geo.interp_by_height(
    wind_day["plume_alt_m"], z800, z700, wind_day["u_800hPa"], wind_day["u_700hPa"])
wind_day["v_plume"], _ = geo.interp_by_height(
    wind_day["plume_alt_m"], z800, z700, wind_day["v_800hPa"], wind_day["v_700hPa"])
wind_day["wind_speed_plume"] = np.hypot(wind_day["u_plume"], wind_day["v_plume"])

print("geopotential height 800 hPa (m):", z800.describe()[["min", "50%", "max"]].round(0).to_dict())
print("geopotential height 700 hPa (m):", z700.describe()[["min", "50%", "max"]].round(0).to_dict())
print("share of hours with plume altitude above 700 hPa (clipped):", round((wind_day["weight_700hPa"] == 1).mean(), 3))

2024-2025 median plume height used to fill missing days: 200 m
wind_day (with plume altitude): 13,312 rows x 11 columns
  missing: none
geopotential height 800 hPa (m): {'min': 1937.0, '50%': 2024.0, 'max': 2079.0}
geopotential height 700 hPa (m): {'min': 3063.0, '50%': 3150.0, 'max': 3200.0}
share of hours with plume altitude above 700 hPa (clipped): 0.05


In [12]:
WIND_LEVELS = {"10m": "10m", "800hPa": "800", "700hPa": "700", "plume": "plume"}
g = wind_day.groupby("date")
wind_daily = pd.DataFrame({"wind_hours": g.size()})
for level, tag in WIND_LEVELS.items():
    _, wind_daily[f"wind_dir_{tag}"] = geo.uv_to_wind(g[f"u_{level}"].mean(), g[f"v_{level}"].mean())
    wind_daily[f"wind_speed_{tag}"] = g[f"wind_speed_{level}"].mean()
wind_daily["plume_alt_m"] = g["plume_alt_m"].first()
wind_daily["weight_700hPa"] = g["weight_700hPa"].mean()

wind_daily["wind_drift_deg"] = (wind_daily["wind_dir_plume"] + 180) % 360
wind_daily["wind_drift_sector"] = geo.deg_to_sector(wind_daily["wind_drift_deg"])
wind_daily = wind_daily.reindex(DATES)

show(wind_daily, "wind_daily")
print("dates with no wind data:", [d.date().isoformat() for d in wind_daily.index[wind_daily["wind_hours"].isna()]])
print(wind_daily["wind_drift_sector"].value_counts().to_string())

wind_daily: 1,025 rows x 13 columns
  missing: {'wind_hours': 1, 'wind_dir_10m': 1, 'wind_speed_10m': 1, 'wind_dir_800': 1, 'wind_speed_800': 1, 'wind_dir_700': 1, 'wind_speed_700': 1, 'wind_dir_plume': 1, 'wind_speed_plume': 1, 'plume_alt_m': 1, 'weight_700hPa': 1, 'wind_drift_deg': 1, 'wind_drift_sector': 1}
dates with no wind data: ['2026-10-21']
wind_drift_sector
W     438
SW    166
E     120
NE     98
NW     88
SE     46
S      36
N      32


## Combine into `volcano_daily` and add lag features

Lags of 1 to 7 days use `shift(k)`, so day t only sees days t-1 to t-7. The 7-day rolling mean covers days t-6 to t
and averages the days that have data.

In [13]:
volcano_daily = pd.concat([so2_daily, eq_daily, plume_daily, wind_daily], axis=1)
volcano_daily["plume_height_filled"] = plume_daily["plume_height_median"].isna().astype(int)
assert volcano_daily.index.equals(DATES)
show(volcano_daily, "volcano_daily (merged)")

LAG_VARS = ["so2_mean", "eq_total", "plume_height_max", "drift_obs_sin", "drift_obs_cos"]
lag_cols = {}
for col in LAG_VARS:
    for k in range(1, 8):
        lag_cols[f"{col}_lag{k}"] = volcano_daily[col].shift(k)
    lag_cols[f"{col}_roll7"] = volcano_daily[col].rolling(7, min_periods=1).mean()
volcano_daily = pd.concat([volcano_daily, pd.DataFrame(lag_cols)], axis=1)
volcano_daily["split"] = np.where(volcano_daily.index <= TRAIN_END, "train", "test")

show(volcano_daily, "volcano_daily (with lags)")
print(volcano_daily["split"].value_counts().to_string())
volcano_daily.loc["2024-09-08":"2024-09-14", ["so2_mean", "so2_mean_lag1", "so2_mean_lag2", "so2_mean_roll7",
                                             "so2_last", "so2_days_since"]]

volcano_daily (merged): 1,025 rows x 40 columns
  missing: {'so2_mean': 170, 'so2_max': 170, 'eq_vt': 8, 'eq_tq': 8, 'eq_tele': 8, 'eq_lfvq': 8, 'eq_tremor': 8, 'eq_e': 8, 'eq_lahar': 8, 'eq_hfvqs': 8, 'eq_total': 8, 'plume_height_max': 142, 'plume_height_median': 142, 'plume_intensity_max': 144, 'plume_intensity_mean': 144, 'drift_obs_sector': 151, 'drift_obs_sin': 151, 'drift_obs_cos': 151, 'wind_hours': 1, 'wind_dir_10m': 1, 'wind_speed_10m': 1, 'wind_dir_800': 1, 'wind_speed_800': 1, 'wind_dir_700': 1, 'wind_speed_700': 1, 'wind_dir_plume': 1, 'wind_speed_plume': 1, 'plume_alt_m': 1, 'weight_700hPa': 1, 'wind_drift_deg': 1, 'wind_drift_sector': 1}
volcano_daily (with lags): 1,025 rows x 81 columns
  missing: {'so2_mean': 170, 'so2_max': 170, 'eq_vt': 8, 'eq_tq': 8, 'eq_tele': 8, 'eq_lfvq': 8, 'eq_tremor': 8, 'eq_e': 8, 'eq_lahar': 8, 'eq_hfvqs': 8, 'eq_total': 8, 'plume_height_max': 142, 'plume_height_median': 142, 'plume_intensity_max': 144, 'plume_intensity_mean': 144, 'drift_obs

,so2_mean,so2_mean_lag1,so2_mean_lag2,so2_mean_roll7,so2_last,so2_days_since
date,,,,,,
2024-09-08,3706.0,1966.0,3557.0,3670.166667,3706.0,0
2024-09-09,2794.0,3706.0,1966.0,3545.000000,2794.0,0
2024-09-10,9985.0,2794.0,3706.0,4103.142857,9985.0,0
2024-09-11,11556.0,9985.0,2794.0,5389.000000,11556.0,0
2024-09-12,10880.0,11556.0,9985.0,6349.142857,10880.0,0
2024-09-13,NaN,10880.0,11556.0,6814.500000,10880.0,1
2024-09-14,NaN,NaN,10880.0,7784.200000,10880.0,2


In [14]:
volcano_daily.reset_index().to_csv(PROCESSED / "volcano_daily.csv", index=False)
print("saved volcano_daily.csv:", volcano_daily.shape)

saved volcano_daily.csv: (1025, 81)


## Barangays within 30 km of the crater

Centroids are computed in EPSG:32651 (UTM zone 51N), then converted to latitude and longitude for the great-circle
distance and bearing from the crater.

In [15]:
import geopandas as gpd
import rasterio

PROVINCE_NAMES = {604500000: "Negros Occidental", 630200000: "Bacolod City", 704600000: "Negros Oriental"}

brgy = gpd.read_file(RAW / "barangays_negros.geojson")
show(brgy, "barangays_negros")
print("CRS:", brgy.crs, "| duplicate adm4_psgc:", brgy["adm4_psgc"].duplicated().sum())
brgy["adm4_psgc"] = brgy["adm4_psgc"].astype("int64")

centroids_utm = brgy.to_crs(32651).geometry.centroid
centroids_ll = centroids_utm.to_crs(4326)
brgy["centroid_x_utm"], brgy["centroid_y_utm"] = centroids_utm.x, centroids_utm.y
brgy["centroid_lat"], brgy["centroid_lon"] = centroids_ll.y, centroids_ll.x
brgy["dist_km"] = geo.haversine_km(geo.CRATER_LAT, geo.CRATER_LON, brgy["centroid_lat"], brgy["centroid_lon"])
brgy["bearing_deg"] = geo.bearing_deg(geo.CRATER_LAT, geo.CRATER_LON, brgy["centroid_lat"], brgy["centroid_lon"])
brgy["bearing_sector"] = geo.deg_to_sector(brgy["bearing_deg"])
brgy["province"] = brgy["adm2_psgc"].map(PROVINCE_NAMES)

study = brgy[brgy["dist_km"] <= STUDY_RADIUS_KM].copy()
show(study.drop(columns="geometry"), f"study barangays (centroid within {STUDY_RADIUS_KM} km)")
print(study.groupby(["adm2_psgc", "province"], dropna=False).size().to_string())
print("distance range (km):", study["dist_km"].min().round(1), "to", study["dist_km"].max().round(1))
assert len(study) == 216, len(study)
assert study["province"].notna().all()

barangays_negros: 1,219 rows x 12 columns
  missing: none
CRS: EPSG:4326 | duplicate adm4_psgc: 0
study barangays (centroid within 30 km): 216 rows x 19 columns
  missing: none
adm2_psgc  province         
604500000  Negros Occidental    163
630200000  Bacolod City           7
704600000  Negros Oriental       46
distance range (km): 4.2 to 30.0


## Elevation at the centroid (SRTM 1 arc-second)

In [16]:
study["elev_m"] = np.nan
for tile in ["N10E122", "N10E123"]:
    path = f"zip://{(RAW / f'{tile}.SRTMGL1.hgt.zip').as_posix()}!{tile}.hgt"
    with rasterio.open(path) as src:
        b = src.bounds
        in_tile = (study["centroid_lon"].between(b.left, b.right, inclusive="left")
                   & study["centroid_lat"].between(b.bottom, b.top, inclusive="right"))
        points = zip(study.loc[in_tile, "centroid_lon"], study.loc[in_tile, "centroid_lat"])
        values = np.array([v[0] for v in src.sample(points)], dtype=float)
        values[values == src.nodata] = np.nan
        study.loc[in_tile, "elev_m"] = values
        print(f"{tile}: {in_tile.sum()} centroids sampled, nodata = {src.nodata}")

show(study[["adm4_psgc", "elev_m"]], "study elevation")
print(study["elev_m"].describe().round(0).to_dict())

N10E122: 77 centroids sampled, nodata = -32768.0
N10E123: 139 centroids sampled, nodata = -32768.0
study elevation: 216 rows x 2 columns
  missing: none
{'count': 216.0, 'mean': 211.0, 'std': 243.0, 'min': 0.0, '25%': 28.0, '50%': 106.0, '75%': 367.0, 'max': 978.0}


## Population (2020 census, single age)

`BgyCode_new` such as "PH0604502001" becomes `adm4_psgc` 604502001 after dropping "PH". Under 5 is ages under 1 to
4, and 60 and over is ages 60 to 79 plus "80 and Over".

In [17]:
UNDER5_COLS = ["Under 1_MF"] + [f"{a}_MF" for a in range(1, 5)]
AGE60_COLS = [f"{a}_MF" for a in range(60, 80)] + ["80 and Over_MF"]

census = pd.read_excel(RAW / "2020-census-total-popn_single-age_malefemale_admin4.xlsx", sheet_name="Brgy_adm4",
                       usecols=["BgyCode_new", "Total_MF"] + UNDER5_COLS + AGE60_COLS)
show(census, "census")
census["adm4_psgc"] = pd.to_numeric(census["BgyCode_new"].str.replace("PH", "", regex=False), errors="coerce")
census = census.dropna(subset=["adm4_psgc"]).astype({"adm4_psgc": "int64"})
show(census, "census (valid code)")
print("duplicate adm4_psgc in census:", census["adm4_psgc"].duplicated().sum())

census["pop_total"] = census["Total_MF"]
census["pop_under5"] = census[UNDER5_COLS].sum(axis=1)
census["pop_60plus"] = census[AGE60_COLS].sum(axis=1)

study = study.merge(census[["adm4_psgc", "pop_total", "pop_under5", "pop_60plus"]],
                    on="adm4_psgc", how="left", validate="1:1")
show(study.drop(columns="geometry"), "study + census")
print("matched:", study["pop_total"].notna().sum(), "of", len(study))
study[["pop_total", "pop_under5", "pop_60plus"]].agg(["sum", "min", "median", "max"])

census: 42,041 rows x 28 columns
  missing: none
census (valid code): 42,041 rows x 29 columns
  missing: none
duplicate adm4_psgc in census: 0
study + census: 216 rows x 23 columns
  missing: none
matched: 216 of 216


,pop_total,pop_under5,pop_60plus
sum,996899.0,102492.0,93625.0
min,125.0,14.0,14.0
median,3109.5,328.0,282.0
max,51863.0,4420.0,5743.0


## Health facilities

Facilities come from the OpenStreetMap export of the Humanitarian OpenStreetMap Team (2026). We keep places where
people can get a respiratory consultation (amenity hospital, clinic, doctors or health_post, or healthcare hospital,
clinic or doctor), one row per OSM id, with polygons as centroids in EPSG:32651. The nearest facility is searched
among all kept facilities, not only those within 30 km, so barangays near the edge are not penalized.

In [18]:
HF_AMENITY = {"hospital", "clinic", "doctors", "health_post"}
HF_HEALTHCARE = {"hospital", "clinic", "doctor"}

hf = gpd.read_file(f"zip://{(RAW / 'hotosm_phl_health_facilities_osm_geojson.zip').as_posix()}!health_facilities.geojson")
show(hf.drop(columns="geometry"), "health_facilities")
hf = hf[hf["amenity"].isin(HF_AMENITY) | hf["healthcare"].isin(HF_HEALTHCARE)].to_crs(32651)
hf["geometry"] = hf.geometry.centroid
show(hf[["amenity", "healthcare", "geometry"]], "health_facilities (kept)")
print("rows sharing an OSM id with another row:", hf["id"].duplicated(keep=False).sum())
hf = hf.drop_duplicates("id")
show(hf[["amenity", "healthcare", "geometry"]], "health_facilities (one row per OSM id)")
print(hf["amenity"].value_counts(dropna=False).to_string())

health_facilities: 19,129 rows x 24 columns
  missing: {'name': 1132, 'name_en': 18553, 'name_fil': 19126, 'amenity': 1959, 'building': 15190, 'healthcare': 5763, 'healthcare_speciality': 16809, 'operator_type': 15708, 'capacity_persons': 18233, 'addr_full': 18993, 'addr_city': 14332, 'source': 19129, 'adm1_pcode': 11, 'adm1_name': 11, 'adm2_pcode': 11, 'adm2_name': 11, 'adm3_pcode': 11, 'adm3_name': 11, 'adm4_pcode': 11, 'adm4_name': 11, 'name_latin': 1121}
health_facilities (kept): 8,203 rows x 3 columns
  missing: {'amenity': 122, 'healthcare': 2794}
rows sharing an OSM id with another row: 534
health_facilities (one row per OSM id): 7,936 rows x 3 columns
  missing: {'amenity': 120, 'healthcare': 2691}
amenity
clinic              2847
hospital            1938
doctors             1527
health_post         1491
NaN                  120
dentist                4
pharmacy               3
community_centre       2
pharmacy;doctors       1
place_of_worship       1
laboratory             1
v

In [19]:
study_pts = gpd.GeoDataFrame(study[["adm4_psgc"]],
                             geometry=gpd.points_from_xy(study["centroid_x_utm"], study["centroid_y_utm"]), crs=32651)
nearest = (gpd.sjoin_nearest(study_pts, hf[["geometry"]], distance_col="hf_dist_m")
           .drop_duplicates("adm4_psgc")[["adm4_psgc", "hf_dist_m"]])
within5 = gpd.sjoin(gpd.GeoDataFrame(study_pts[["adm4_psgc"]], geometry=study_pts.buffer(5000)),
                    hf[["geometry"]], predicate="contains").groupby("adm4_psgc").size().rename("hf_within_5km")

study = study.merge(nearest, on="adm4_psgc", how="left", validate="1:1")
study["hf_nearest_km"] = study.pop("hf_dist_m") / 1000
study = study.merge(within5, on="adm4_psgc", how="left", validate="1:1")
study["hf_within_5km"] = study["hf_within_5km"].fillna(0).astype(int)
show(study.drop(columns="geometry"), "study + health facilities")
study[["hf_nearest_km", "hf_within_5km"]].describe().round(2)

study + health facilities: 216 rows x 25 columns
  missing: none


,hf_nearest_km,hf_within_5km
count,216.00,216.00
mean,6.75,1.14
std,5.35,1.57
min,0.13,0.00
25%,2.58,0.00
50%,5.33,0.00
75%,9.06,2.00
max,22.59,6.00


## Save `barangays_study_area`

In [20]:
BRGY_COLUMNS = ["adm4_psgc", "adm4_en", "adm3_psgc", "adm2_psgc", "province", "muni_code",
                "centroid_lat", "centroid_lon", "centroid_x_utm", "centroid_y_utm",
                "dist_km", "bearing_deg", "bearing_sector", "elev_m",
                "pop_total", "pop_under5", "pop_60plus", "hf_nearest_km", "hf_within_5km"]
barangays = study[BRGY_COLUMNS].sort_values("adm4_psgc").reset_index(drop=True)
show(barangays, "barangays_study_area")
barangays.to_csv(PROCESSED / "barangays_study_area.csv", index=False)
print("saved barangays_study_area.csv:", barangays.shape)
barangays.head()

barangays_study_area: 216 rows x 19 columns
  missing: none
saved barangays_study_area.csv: (216, 19)


,adm4_psgc,adm4_en,adm3_psgc,adm2_psgc,province,muni_code,centroid_lat,centroid_lon,centroid_x_utm,centroid_y_utm,dist_km,bearing_deg,bearing_sector,elev_m,pop_total,pop_under5,pop_60plus,hf_nearest_km,hf_within_5km
0,604502001,Abuanan,604502000,604500000,Negros Occidental,604502000,10.530164,122.986652,498539.589116,1.164030e+06,20.620864,309.595110,NW,40.0,5951,660,530,4.371090,2
1,604502002,Alianza,604502000,604500000,Negros Occidental,604502000,10.473681,122.926378,491943.340420,1.157786e+06,23.508182,286.981671,W,24.0,3170,294,297,4.724503,1
2,604502003,Atipuluan,604502000,604500000,Negros Occidental,604502000,10.511075,122.957420,495340.908743,1.161920e+06,22.040501,300.004910,NW,32.0,3944,382,375,4.215313,1
3,604502004,Bacong-Montilla,604502000,604500000,Negros Occidental,604502000,10.522859,123.047954,505246.908122,1.163223e+06,15.375618,323.302341,NW,96.0,8219,1025,771,5.199265,0
4,604502005,Bagroy,604502000,604500000,Negros Occidental,604502000,10.478912,122.876258,486458.725800,1.158366e+06,28.938834,284.921508,W,12.0,1684,181,182,6.850403,0


## Exposure labels from the report log

Each report log row is a team-verified sulfur odor or ashfall report for one barangay on one event date, joined by
`psgc_code` and `event_date`, never by name. `exposed` marks any report (`exposed_ash` and `exposed_odor` by hazard),
and `report_day` = 1 on every event date in the log, reports outside 30 km included, so barangays without a report
on those days are known negatives.

In [21]:
report_log = pd.read_csv(REPORT_LOG)
show(report_log, "report_log")
report_log["psgc_code"] = report_log["psgc_code"].astype("Int64")
report_log["event_date"] = pd.to_datetime(report_log["event_date"])
print(report_log["hazard"].value_counts().to_string())

report_log["is_ash"] = report_log["hazard"].str.contains("ashfall").astype(int)
report_log["is_odor"] = report_log["hazard"].str.contains("sulfur odor").astype(int)
report_dates = pd.DatetimeIndex(sorted(report_log["event_date"].unique()))
print("event dates:", len(report_dates), [d.date().isoformat() for d in report_dates])

in_study = report_log["psgc_code"].isin(barangays["adm4_psgc"])
print("rows with no psgc_code:", report_log["psgc_code"].isna().sum())
print("rows outside the 216 study barangays:", (~in_study & report_log["psgc_code"].notna()).sum())
print("log within_30km vs our selection:\n", pd.crosstab(report_log["within_30km"], in_study,
                                                        rownames=["within_30km"], colnames=["in_study"]))

reports_study = report_log[in_study]
show(reports_study, "report_log (study barangays)")
labels = (reports_study.groupby(["psgc_code", "event_date"])
          .agg(exposed=("hazard", "size"), exposed_ash=("is_ash", "max"), exposed_odor=("is_odor", "max"))
          .reset_index()
          .rename(columns={"psgc_code": "adm4_psgc", "event_date": "date"})
          .astype({"adm4_psgc": "int64"}))
labels["exposed"] = 1
show(labels, "labels (barangay-days)")
print(labels.groupby("date")[["exposed", "exposed_ash", "exposed_odor"]].sum().to_string())

report_log: 243 rows x 14 columns
  missing: {'event_date_note': 177, 'barangay_psgc_name': 1, 'psgc_code': 1, 'km_from_crater': 1, 'note': 119, 'checked_by': 243}
hazard
ashfall                    124
sulfur odor                 78
ashfall and sulfur odor     41
event dates: 14 ['2024-06-03', '2024-09-11', '2024-09-24', '2024-10-28', '2024-11-15', '2024-12-09', '2025-02-15', '2025-04-08', '2025-10-24', '2026-02-19', '2026-02-26', '2026-03-15', '2026-07-09', '2026-10-10']
rows with no psgc_code: 1
rows outside the 216 study barangays: 70
log within_30km vs our selection:
 in_study     False  True 
within_30km              
False           70      1
True             1    171
report_log (study barangays): 172 rows x 16 columns
  missing: {'event_date_note': 118, 'note': 86, 'checked_by': 172}
labels (barangay-days): 166 rows x 5 columns
  missing: none
            exposed  exposed_ash  exposed_odor
date                                          
2024-06-03        7            0           

## Barangay x date panel

`angle_diff_deg` is the smallest angle (0 to 180) between the barangay's bearing from the crater and the
plume-level wind drift; 0 means the barangay is directly downwind.

In [22]:
panel = pd.MultiIndex.from_product([barangays["adm4_psgc"], DATES], names=["adm4_psgc", "date"]).to_frame(index=False)
show(panel, "panel (grid)")

panel = panel.merge(barangays, on="adm4_psgc", how="left", validate="m:1")
show(panel, "panel + barangays")
panel = panel.merge(volcano_daily.reset_index(), on="date", how="left", validate="m:1")
show(panel, "panel + volcano_daily")
panel = panel.merge(labels, on=["adm4_psgc", "date"], how="left", validate="1:1")
panel[["exposed", "exposed_ash", "exposed_odor"]] = panel[["exposed", "exposed_ash", "exposed_odor"]].fillna(0).astype(int)
panel["report_day"] = panel["date"].isin(report_dates).astype(int)
panel["angle_diff_deg"] = geo.angle_diff_deg(panel["bearing_deg"], panel["wind_drift_deg"])
show(panel, "panel + labels")

assert len(panel) == 216 * 1025 == 221_400
assert panel["exposed"].sum() == len(labels)

panel (grid): 221,400 rows x 2 columns
  missing: none
panel + barangays: 221,400 rows x 20 columns
  missing: none


panel + volcano_daily: 221,400 rows x 101 columns
  missing: {'so2_mean': 36720, 'so2_max': 36720, 'eq_vt': 1728, 'eq_tq': 1728, 'eq_tele': 1728, 'eq_lfvq': 1728, 'eq_tremor': 1728, 'eq_e': 1728, 'eq_lahar': 1728, 'eq_hfvqs': 1728, 'eq_total': 1728, 'plume_height_max': 30672, 'plume_height_median': 30672, 'plume_intensity_max': 31104, 'plume_intensity_mean': 31104, 'drift_obs_sector': 32616, 'drift_obs_sin': 32616, 'drift_obs_cos': 32616, 'wind_hours': 216, 'wind_dir_10m': 216, 'wind_speed_10m': 216, 'wind_dir_800': 216, 'wind_speed_800': 216, 'wind_dir_700': 216, 'wind_speed_700': 216, 'wind_dir_plume': 216, 'wind_speed_plume': 216, 'plume_alt_m': 216, 'weight_700hPa': 216, 'wind_drift_deg': 216, 'wind_drift_sector': 216, 'so2_mean_lag1': 36936, 'so2_mean_lag2': 37152, 'so2_mean_lag3': 37368, 'so2_mean_lag4': 37584, 'so2_mean_lag5': 37800, 'so2_mean_lag6': 38016, 'so2_mean_lag7': 38232, 'eq_total_lag1': 1944, 'eq_total_lag2': 2160, 'eq_total_lag3': 2376, 'eq_total_lag4': 2592, 'eq_tot

In [23]:
first_cols = ["adm4_psgc", "date", "split", "exposed", "exposed_ash", "exposed_odor", "report_day", "angle_diff_deg"]
panel = panel[first_cols + [c for c in panel.columns if c not in first_cols]]
panel.to_csv(PROCESSED / "barangay_daily_panel.csv", index=False)
print("saved barangay_daily_panel.csv:", panel.shape,
      f"| {(PROCESSED / 'barangay_daily_panel.csv').stat().st_size / 1e6:.0f} MB")

saved barangay_daily_panel.csv: (221400, 106) | 219 MB


## Validation

In [24]:
print("Row counts")
print(f"  volcano_daily:        {len(volcano_daily):>7,}  (expected 1,025)")
print(f"  barangays_study_area: {len(barangays):>7,}  (expected 216)")
print(f"  barangay_daily_panel: {len(panel):>7,}  (expected 221,400)")


def missing_table(df):
    """Missing values and their share (%) per column, for columns with any."""
    na = df.isna().sum()
    return pd.DataFrame({"missing": na, "pct": (100 * na / len(df)).round(1)})[na > 0]


for name, df in [("volcano_daily", volcano_daily), ("barangays_study_area", barangays), ("barangay_daily_panel", panel)]:
    mt = missing_table(df)
    print(f"\nMissing values: {name} ({df.shape[1]} columns, {df.shape[1] - len(mt)} complete)")
    print(mt.to_string() if len(mt) else "  none")

Row counts
  volcano_daily:          1,025  (expected 1,025)
  barangays_study_area:     216  (expected 216)
  barangay_daily_panel: 221,400  (expected 221,400)

Missing values: volcano_daily (81 columns, 12 complete)
                        missing   pct
so2_mean                    170  16.6
so2_max                     170  16.6
eq_vt                         8   0.8
eq_tq                         8   0.8
eq_tele                       8   0.8
eq_lfvq                       8   0.8
eq_tremor                     8   0.8
eq_e                          8   0.8
eq_lahar                      8   0.8
eq_hfvqs                      8   0.8
eq_total                      8   0.8
plume_height_max            142  13.9
plume_height_median         142  13.9
plume_intensity_max         144  14.0
plume_intensity_mean        144  14.0
drift_obs_sector            151  14.7
drift_obs_sin               151  14.7
drift_obs_cos               151  14.7
wind_hours                    1   0.1
wind_dir_10m          

In [25]:
top = volcano_daily["so2_max"].idxmax()
print(f"Max SO2: {volcano_daily['so2_max'].max():,.0f} t/d on {top.date()}  (expected 11,556 on 2024-09-11)")
print(f"Days with SO2: {volcano_daily['so2_observed'].sum()}  (expected about 855)")
print(f"Readings equal to 0 t/d: {(so2_raw['emission'] == 0).sum()}")

drift_days = volcano_daily["drift_obs_sector"].dropna()
drift_obs = steam_raw["drift_sector"].dropna()
print(f"\nSW drift share: {(drift_days == 'SW').mean():.1%} of {len(drift_days)} days (daily mode), "
      f"{(drift_obs == 'SW').mean():.1%} of {len(drift_obs)} single observations  (expected about 37%)")
for split, d in volcano_daily.groupby("split"):
    s = d["drift_obs_sector"].dropna()
    print(f"  {split}: {(s == 'SW').mean():.1%} of {len(s)} days")

Max SO2: 11,556 t/d on 2024-09-11  (expected 11,556 on 2024-09-11)
Days with SO2: 855  (expected about 855)
Readings equal to 0 t/d: 2

SW drift share: 34.1% of 874 days (daily mode), 37.6% of 2523 single observations  (expected about 37%)
  test: 35.5% of 265 days
  train: 33.5% of 609 days


In [26]:
exposure_counts = panel.groupby("split").agg(
    barangay_days=("exposed", "size"),
    exposed=("exposed", "sum"),
    exposed_ash=("exposed_ash", "sum"),
    exposed_odor=("exposed_odor", "sum"),
    report_days=("report_day", lambda s: panel.loc[s.index[s == 1], "date"].nunique()),
    barangays_ever_exposed=("exposed", lambda s: panel.loc[s.index[s == 1], "adm4_psgc"].nunique()),
)
print("Exposure labels by split")
exposure_counts

Exposure labels by split


,barangay_days,exposed,exposed_ash,exposed_odor,report_days,barangays_ever_exposed
split,,,,,,
test,63504,92,78,38,5,67
train,157896,74,33,56,9,46


In [27]:
both = volcano_daily.dropna(subset=["drift_obs_sector", "wind_drift_sector"]).copy()
both["sector_gap"] = geo.angle_diff_deg(both["drift_obs_sector"].map(geo.SECTOR_DEG),
                                        both["wind_drift_sector"].map(geo.SECTOR_DEG)) / 45
for tag in ["10m", "800", "700"]:
    both[f"sector_{tag}"] = geo.deg_to_sector((both[f"wind_dir_{tag}"] + 180) % 360)

rows = []
for split, d in [("all", both)] + list(both.groupby("split")):
    row = {"split": split, "days": len(d),
           "plume level: same sector": (d["sector_gap"] == 0).mean(),
           "plume level: within 1 sector": (d["sector_gap"] <= 1).mean()}
    for tag in ["10m", "800", "700"]:
        row[f"{tag}: same sector"] = (d[f"sector_{tag}"] == d["drift_obs_sector"]).mean()
    rows.append(row)
print("Same-day agreement between wind-based drift sector and observed drift sector")
pd.DataFrame(rows).set_index("split").round(3)

Same-day agreement between wind-based drift sector and observed drift sector


,days,plume level: same sector,plume level: within 1 sector,10m: same sector,800: same sector,700: same sector
split,,,,,,
all,873,0.447,0.885,0.315,0.447,0.427
test,264,0.492,0.939,0.231,0.447,0.470
train,609,0.427,0.862,0.351,0.447,0.409
